# 🔬 Kaggle Notebook 3 — Quantum Interpretability Visualizations
## Understanding the Quantum Advantage — Research Figures

**Secrets required:** `HF_TOKEN_1`, `HF_TOKEN_2`

All figures pushed to `Shanmuk4622/quantum-lpr-checkpoints` → `results/`

In [ ]:
!pip install pennylane pennylane-lightning huggingface_hub seaborn editdistance -q

import os, json, zipfile
import torch, torch.nn as nn
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, random_split
import pennylane as qml
from huggingface_hub import HfApi, login, whoami, hf_hub_download

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')

In [ ]:
# ── HF Auth + Data Setup (same pattern as other notebooks) ─
try:
    from kaggle_secrets import UserSecretsClient
    s=UserSecretsClient()
    HF_TOKENS=[s.get_secret('HF_TOKEN_1'),s.get_secret('HF_TOKEN_2')]
except Exception:
    HF_TOKENS=['ADD_YOUR_HF_TOKEN_1_IN_KAGGLE_SECRETS','ADD_YOUR_HF_TOKEN_2_IN_KAGGLE_SECRETS']

HF_USERNAME='Shanmuk4622'
HF_DATASET_REPO=f'{HF_USERNAME}/quantum-lpr-dataset'
HF_MODEL_REPO=f'{HF_USERNAME}/quantum-lpr-checkpoints'
active_token=None; api=None
for tok in HF_TOKENS:
    try:
        login(token=tok,add_to_git_credential=False); info=whoami(token=tok)
        active_token=tok; api=HfApi(token=tok); print(f'✅ HF: {info["name"]}'); break
    except Exception: pass

WORK_DIR='/kaggle/working'; DATA_DIR=f'{WORK_DIR}/lpr_data'; CKPT_DIR=f'{WORK_DIR}/ckpt'
RESULTS_DIR=f'{WORK_DIR}/results'
os.makedirs(DATA_DIR,exist_ok=True); os.makedirs(CKPT_DIR,exist_ok=True); os.makedirs(RESULTS_DIR,exist_ok=True)
ZIP_L=f'{DATA_DIR}/train.zip'; CSV_L=f'{DATA_DIR}/labels.csv'; EXT_DIR=f'{DATA_DIR}/images'
CHARS='0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
CHAR2IDX={c:i+1 for i,c in enumerate(CHARS)}; IDX2CHAR={i+1:c for i,c in enumerate(CHARS)}
N_QUBITS=8; N_LAYERS=2; SEED=42

for hfn,loc in [('data/2_train_hr_images.csv',CSV_L),('data/wYe7pBJ7-train.zip',ZIP_L)]:
    if not os.path.exists(loc):
        hf_hub_download(repo_id=HF_DATASET_REPO,filename=hfn,repo_type='dataset',
                        token=active_token,local_dir=DATA_DIR,local_dir_use_symlinks=False)
if not os.path.exists(EXT_DIR):
    with zipfile.ZipFile(ZIP_L) as z: z.extractall(EXT_DIR)
print('✅ Data + Auth ready')

In [ ]:
# ── Model + Load ─────────────────────────────────────────
class ZeroDCE_Light(nn.Module):
    def __init__(self): super().__init__(); self.conv=nn.Sequential(nn.Conv2d(3,16,3,1,1),nn.ReLU(),nn.Conv2d(16,16,3,1,1),nn.ReLU(),nn.Conv2d(16,24,3,1,1),nn.Tanh())
    def forward(self,x):
        A,e=self.conv(x),x
        for i in range(8): e=e+A[:,3*i:3*(i+1)]*(torch.pow(e,2)-e)
        return e

d_qml=qml.device('default.qubit',wires=N_QUBITS)
@qml.qnode(d_qml,interface='torch')
def qcirc(inputs,weights):
    qml.templates.AngleEmbedding(inputs,wires=range(N_QUBITS))
    qml.templates.StronglyEntanglingLayers(weights,wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

class QL(nn.Module):
    def __init__(self): super().__init__(); self.q=qml.qnn.TorchLayer(qcirc,{'weights':(N_LAYERS,N_QUBITS,3)})
    def forward(self,x): return self.q(x)

class HybridLPRNet_8Q(nn.Module):
    def __init__(self,nc=37):
        super().__init__()
        self.enhancer=ZeroDCE_Light()
        self.features=nn.Sequential(nn.Conv2d(3,64,3,1,1),nn.MaxPool2d(2),nn.ReLU(),nn.Conv2d(64,128,3,1,1),nn.MaxPool2d(2),nn.ReLU(),nn.Conv2d(128,N_QUBITS,1,1))
        self.quantum=QL(); self.rnn=nn.LSTM(N_QUBITS,128,bidirectional=True,batch_first=True); self.cls=nn.Linear(256,nc)
    def forward(self,x):
        x=self.enhancer(x); x=self.features(x); b,c,h,w=x.size()
        xs=x.mean(dim=2).permute(0,2,1); q=self.quantum(xs.reshape(-1,N_QUBITS)).reshape(b,w,N_QUBITS)
        return self.cls(self.rnn(q)[0]).permute(1,0,2)
    def pre_q(self,x):
        with torch.no_grad(): x=self.enhancer(x); x=self.features(x); return x.mean(dim=2).permute(0,2,1)
    def post_q(self,x):
        with torch.no_grad():
            pre=self.pre_q(x); b,w,_=pre.shape
            return self.quantum(pre.reshape(-1,N_QUBITS)).reshape(b,w,N_QUBITS)

q_model=HybridLPRNet_8Q(37).to(device)
for hfp in ['quantum/best.pth','quantum/latest.pth']:
    try:
        f=hf_hub_download(repo_id=HF_MODEL_REPO,filename=hfp,repo_type='model',
                          token=active_token,local_dir=CKPT_DIR,local_dir_use_symlinks=False,force_download=True)
        ck=torch.load(f,map_location=device); q_model.load_state_dict(ck['model_state_dict'])
        print(f'✅ Quantum model loaded: {hfp}'); break
    except Exception as e: print(f'  ⚠️  {e}')
q_model.eval()

# Dataset
class LPRDataset(Dataset):
    def __init__(self,csv,root,tf):
        self.df=pd.read_csv(csv); self.root=root; self.tf=tf
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; p=str(r['path'])
        f=p if p.startswith('/') else os.path.join(self.root,p)
        if not os.path.exists(f): f=os.path.join(EXT_DIR,'train',os.path.basename(p))
        try: img=Image.open(f).convert('RGB')
        except: img=Image.new('RGB',(256,64))
        if self.tf: img=self.tf(img)
        return img, str(r['label']).upper()

tf=transforms.Compose([transforms.Resize((64,256)),transforms.ToTensor()])
dataset=LPRDataset(CSV_L,EXT_DIR,tf)
print(f'✅ Dataset: {len(dataset)} samples')

In [ ]:
# ── VIZ 1: Qubit Heatmaps ─────────────────────────────
np.random.seed(SEED)
sample_idx=np.random.choice(len(dataset),3,replace=False)

for si in sample_idx:
    img,lbl=dataset[int(si)]
    inp=img.unsqueeze(0).to(device)
    pre =q_model.pre_q(inp).cpu().squeeze(0).numpy()   # [W,8]
    post=q_model.post_q(inp).cpu().squeeze(0).numpy()  # [W,8]

    fig,axes=plt.subplots(2,1,figsize=(16,5))
    fig.suptitle(f'Qubit Activations — Label: {lbl}',fontsize=13,fontweight='bold')
    im0=axes[0].imshow(pre.T,aspect='auto',cmap='viridis')
    axes[0].set_title('CNN → Quantum Input (pre-circuit)',fontsize=11)
    axes[0].set_ylabel('Qubit'); axes[0].set_xlabel('Width position'); axes[0].set_yticks(range(N_QUBITS))
    plt.colorbar(im0,ax=axes[0])
    im1=axes[1].imshow(post.T,aspect='auto',cmap='magma')
    axes[1].set_title('Quantum Output ⟨Z_i⟩ (post-circuit)',fontsize=11)
    axes[1].set_ylabel('Qubit'); axes[1].set_xlabel('Width position'); axes[1].set_yticks(range(N_QUBITS))
    plt.colorbar(im1,ax=axes[1])
    plt.tight_layout()
    out=os.path.join(RESULTS_DIR,f'qubit_heatmap_{si}.png')
    plt.savefig(out,dpi=130,bbox_inches='tight'); plt.show()
    api.upload_file(path_or_fileobj=out,path_in_repo=f'results/qubit_heatmap_{si}.png',
                    repo_id=HF_MODEL_REPO,repo_type='model',token=active_token)

print('✅ Qubit heatmaps done & pushed to HF')

In [ ]:
# ── VIZ 2: Confusable Character Fingerprints ───────────
pairs=[('0','O'),('1','I'),('5','S'),('8','B')]
fps={c:[] for p in pairs for c in p}

for i in range(min(600,len(dataset))):
    img,lbl=dataset[i]
    for ch in fps:
        if ch in lbl and len(fps[ch])<25:
            with torch.no_grad():
                pq=q_model.post_q(img.unsqueeze(0).to(device)).cpu().squeeze(0).numpy()
            fps[ch].append(pq.mean(axis=0))

fig,axes=plt.subplots(2,2,figsize=(14,10))
fig.suptitle('Quantum Qubit Fingerprints: Confusable Character Pairs',fontsize=14,fontweight='bold')
cols=[('#E91E63','#FF9800'),('#2196F3','#4CAF50'),('#9C27B0','#00BCD4'),('#F44336','#8BC34A')]
for ax,(c1,c2),(col1,col2) in zip(axes.flatten(),pairs,cols):
    m1=np.array(fps[c1]).mean(axis=0) if fps[c1] else np.zeros(N_QUBITS)
    m2=np.array(fps[c2]).mean(axis=0) if fps[c2] else np.zeros(N_QUBITS)
    s1=np.array(fps[c1]).std(axis=0)  if fps[c1] else np.zeros(N_QUBITS)
    s2=np.array(fps[c2]).std(axis=0)  if fps[c2] else np.zeros(N_QUBITS)
    x=np.arange(N_QUBITS); w=0.35
    ax.bar(x-w/2,m1,w,color=col1,alpha=0.8,label=f"'{c1}'",yerr=s1)
    ax.bar(x+w/2,m2,w,color=col2,alpha=0.8,label=f"'{c2}'",yerr=s2)
    ax.set_xticks(x); ax.set_xticklabels([f'Q{i}' for i in range(N_QUBITS)])
    ax.set_title(f"'{c1}' vs '{c2}'"); ax.set_ylabel('Mean ⟨Z⟩'); ax.legend(); ax.grid(axis='y',alpha=0.3)
plt.tight_layout()
cf_out=os.path.join(RESULTS_DIR,'char_confusion_qubits.png')
plt.savefig(cf_out,dpi=130,bbox_inches='tight'); plt.show()
api.upload_file(path_or_fileobj=cf_out,path_in_repo='results/char_confusion_qubits.png',
                repo_id=HF_MODEL_REPO,repo_type='model',token=active_token)
print('✅ Character confusion analysis pushed to HF')

In [ ]:
# ── VIZ 3: ZeroDCE Quality at Different Darkness Levels ─
gammas=[1.5,2.0,2.5,3.0,3.5]
img,lbl=dataset[int(sample_idx[1])]
fig,axes=plt.subplots(2,len(gammas)+1,figsize=(22,5))
fig.suptitle(f'ZeroDCE Enhancement vs Darkness — Label: {lbl}',fontsize=13,fontweight='bold')
axes[0,0].imshow(img.permute(1,2,0).numpy()); axes[0,0].set_title('Original',fontweight='bold'); axes[0,0].axis('off')
axes[1,0].axis('off'); axes[1,0].text(0.5,0.5,'γ=1.0\n(clean)',ha='center',va='center',bbox=dict(boxstyle='round',fc='#d4edda'))
for col,g in enumerate(gammas):
    dark=torch.clamp(torch.pow(img,g)+torch.randn_like(img)*0.05,0,1)
    with torch.no_grad(): enh=q_model.enhancer(dark.unsqueeze(0).to(device)).cpu().squeeze(0)
    axes[0,col+1].imshow(np.clip(enh.permute(1,2,0).numpy(),0,1)); axes[0,col+1].set_title(f'Enhanced γ={g}',fontsize=9); axes[0,col+1].axis('off')
    axes[1,col+1].imshow(np.clip(dark.permute(1,2,0).numpy(),0,1)); axes[1,col+1].set_title(f'Dark Input γ={g}',fontsize=9,color='gray'); axes[1,col+1].axis('off')
plt.tight_layout()
zd_out=os.path.join(RESULTS_DIR,'zero_dce_quality.png')
plt.savefig(zd_out,dpi=130,bbox_inches='tight'); plt.show()
api.upload_file(path_or_fileobj=zd_out,path_in_repo='results/zero_dce_quality.png',
                repo_id=HF_MODEL_REPO,repo_type='model',token=active_token)
print('✅ ZeroDCE quality chart pushed to HF')

In [ ]:
# ── VIZ 4: Training Curves (from HF history JSON) ───────
def load_hist(hf_path):
    try:
        f=hf_hub_download(repo_id=HF_MODEL_REPO,filename=hf_path,repo_type='model',
                          token=active_token,local_dir=CKPT_DIR,local_dir_use_symlinks=False,force_download=True)
        with open(f) as fp: return json.load(fp)
    except: return None

qh=load_hist('quantum/history.json'); ch=load_hist('classical/history.json')

if qh and ch:
    fig,axes=plt.subplots(1,3,figsize=(18,5))
    fig.suptitle('Full Training History — Quantum ⚡ vs Classical 🔷',fontsize=14,fontweight='bold')
    def pm(ax,qd,cd,title,yl):
        qe=range(1,len(qd)+1); ce=range(1,len(cd)+1)
        ax.plot(qe,qd,'#7B2D8B',lw=2,label='Quantum ⚡')
        ax.plot(ce,cd,'#2196F3',lw=2,label='Classical 🔷',ls='--')
        ax.fill_between(qe,qd,alpha=0.1,color='#7B2D8B')
        ax.fill_between(ce,cd,alpha=0.1,color='#2196F3')
        ax.set_title(title,fontweight='bold'); ax.set_xlabel('Epoch'); ax.set_ylabel(yl); ax.legend(); ax.grid(alpha=0.3)
    pm(axes[0],qh['val_loss'],ch['val_loss'],'Validation Loss','CTC Loss')
    pm(axes[1],qh['val_cer'],ch['val_cer'],'Character Error Rate','CER ↓')
    pm(axes[2],[1-v for v in qh['val_wer']],[1-v for v in ch['val_wer']],'Plate Accuracy','Acc ↑')
    plt.tight_layout()
    tc_out=os.path.join(RESULTS_DIR,'full_training_curves.png')
    plt.savefig(tc_out,dpi=150,bbox_inches='tight'); plt.show()
    api.upload_file(path_or_fileobj=tc_out,path_in_repo='results/full_training_curves.png',
                    repo_id=HF_MODEL_REPO,repo_type='model',token=active_token)
    print('✅ Training curves pushed to HF')
else:
    print('⚠️  No history found — run Notebook 1 first')

In [ ]:
# ── VIZ 5: Architecture Diagram ─────────────────────────
fig,ax=plt.subplots(1,1,figsize=(16,4)); ax.axis('off')
stages=[('Night\nInput','#455A64'),('ZeroDCE\nEnhancer','#1565C0'),
        ('CNN\n(→8ch)','#1B5E20'),('8-Qubit\nCircuit','#6A1B9A'),
        ('Bi-LSTM','#E65100'),('CTC\nDecode','#BF360C')]
xs=np.linspace(0.06,0.94,len(stages))
for x,(lbl,col) in zip(xs,stages):
    ax.add_patch(plt.FancyBboxPatch((x-0.07,0.18),0.13,0.64,boxstyle='round,pad=0.01',
                 facecolor=col,edgecolor='white',linewidth=2,transform=ax.transAxes,clip_on=False))
    ax.text(x,0.5,lbl,ha='center',va='center',color='white',fontsize=10,fontweight='bold',transform=ax.transAxes)
for i in range(len(xs)-1):
    ax.annotate('',xy=(xs[i+1]-0.07,0.5),xytext=(xs[i]+0.07,0.5),xycoords='axes fraction',textcoords='axes fraction',
                arrowprops=dict(arrowstyle='->',color='#333',lw=2.5))
ax.text(0.485,0.04,'256-dim Hilbert Space  •  AngleEmbedding  •  StronglyEntanglingLayers',
        ha='center',va='bottom',color='#7B2D8B',fontsize=9,fontweight='bold',transform=ax.transAxes)
ax.set_title('HybridLPRNet_8Q — Quantum-Enhanced License Plate Recognition Pipeline',
             fontsize=13,fontweight='bold',pad=20)
arch_out=os.path.join(RESULTS_DIR,'architecture_diagram.png')
plt.savefig(arch_out,dpi=150,bbox_inches='tight',facecolor='white'); plt.show()
api.upload_file(path_or_fileobj=arch_out,path_in_repo='results/architecture_diagram.png',
                repo_id=HF_MODEL_REPO,repo_type='model',token=active_token)
print('✅ Architecture diagram pushed to HF')
print()
print('🎉 All visualizations complete and pushed to:')
print(f'   https://huggingface.co/Shanmuk4622/quantum-lpr-checkpoints/tree/main/results/')